In [1]:
import subprocess
import os
import trimesh
import numpy as np
from utils_numba import compute_mvc, mvc_3d_single
# import mlx.core as mx

import igl

from utils_1 import RemappedCageFile, obj_to_off
from utils_2 import RemappedCageFolder, average_edge_length

import shutil
from pathlib import Path
import subprocess, os

from meshplot import plot

import numpy as np
from numba import njit, prange
from numba.typed import Dict
from numba import types

import time


In [2]:
action = "jumping"
alpha = 0.25
alpha_str = str(alpha).replace(".", "_")
input_file = f"avg_meshes/{action}Avg.obj"
avg_string = os.path.basename(input_file).replace('.obj', '')



In [3]:
cage_avg_folder = f"Dynamic mesh codec for mac v2/decompressed_{action}_1/{alpha_str}"
cage_files = sorted([f for f in os.listdir(cage_avg_folder) if f.endswith(".obj")])
cage_vertices_list = []
for cage_file in cage_files:
    cage_path = os.path.join(cage_avg_folder, cage_file)
    cage_V, cage_F = igl.read_triangle_mesh(cage_path)
    cage_vertices_list.append(cage_V)

U_seq_np = np.array(cage_vertices_list, dtype=np.float64)  # (m, N, 3)


In [4]:
U_seq_np.shape

(150, 2502, 3)

In [5]:
original_mesh_folder = f"original_meshes/{action}_mesh"
original_mesh_files = sorted([f for f in os.listdir(original_mesh_folder) if f.endswith(".obj")])
original_mesh_vertices_list = []
for original_mesh_file in original_mesh_files:
    original_mesh_path = os.path.join(original_mesh_folder, original_mesh_file)
    original_mesh_V, original_mesh_F = igl.read_triangle_mesh(original_mesh_path)
    original_mesh_vertices_list.append(original_mesh_V)

V_seq_np = np.array(original_mesh_vertices_list, dtype=np.float64)  # (m, N, 3)
V_seq_np.shape

  o mesh_0001.off
  o mesh_0002.off
  o mesh_0003.off
  o mesh_0004.off
  o mesh_0005.off
  o mesh_0006.off
  o mesh_0007.off
  o mesh_0008.off
  o mesh_0009.off
  o mesh_0010.off
  o mesh_0011.off
  o mesh_0012.off
  o mesh_0013.off
  o mesh_0014.off
  o mesh_0015.off
  o mesh_0016.off
  o mesh_0017.off
  o mesh_0018.off
  o mesh_0019.off
  o mesh_0020.off
  o mesh_0021.off
  o mesh_0022.off
  o mesh_0023.off
  o mesh_0024.off
  o mesh_0025.off
  o mesh_0026.off
  o mesh_0027.off
  o mesh_0028.off
  o mesh_0029.off
  o mesh_0030.off
  o mesh_0031.off
  o mesh_0032.off
  o mesh_0033.off
  o mesh_0034.off
  o mesh_0035.off
  o mesh_0036.off
  o mesh_0037.off
  o mesh_0038.off
  o mesh_0039.off
  o mesh_0040.off
  o mesh_0041.off
  o mesh_0042.off
  o mesh_0043.off
  o mesh_0044.off
  o mesh_0045.off
  o mesh_0046.off
  o mesh_0047.off
  o mesh_0048.off
  o mesh_0049.off
  o mesh_0050.off
  o mesh_0051.off
  o mesh_0052.off
  o mesh_0053.off
  o mesh_0054.off
  o mesh_0055.off
  o mesh_0

(150, 10002, 3)

In [16]:
input_file.replace('.obj', '_decoded_restructured.obj')

'avg_meshes/jumpingAvg_decoded_restructured.obj'

In [6]:
mesh_vertices, mesh_faces = igl.read_triangle_mesh(input_file.replace('.obj', '_decoded_restructured.obj'))


cage_vertices, cage_faces = igl.read_triangle_mesh(input_file.replace('.obj', '_cage_decoded_restructured.obj'))


In [18]:
mesh_vertices.shape, mesh_faces.shape, cage_vertices.shape, cage_faces.shape

((10002, 3), (20000, 3), (2502, 3), (5000, 3))

In [20]:
mvc = compute_mvc(mesh_vertices, cage_vertices, cage_faces)

In [7]:
num_vertices = mesh_vertices.shape[0]
refined_mesh_vertices = np.zeros_like(mesh_vertices, dtype=np.float64)
# loss_per_vertex = np.zeros(num_vertices, dtype=np.float64)

mesh_vertices_c = np.ascontiguousarray(mesh_vertices, dtype=np.float64)
U_seq_c         = np.ascontiguousarray(U_seq_np, dtype=np.float64)
V_seq_c         = np.ascontiguousarray(V_seq_np, dtype=np.float64)
cage_V_c        = np.ascontiguousarray(cage_vertices, dtype=np.float64)
cage_F_c        = np.ascontiguousarray(cage_faces, dtype=np.int32)   # faces int32
refined_mesh_vertices = np.empty_like(mesh_vertices_c)



In [8]:
V_seq_c.shape

(150, 10002, 3)

In [ ]:
@njit(cache=True, fastmath=False)
def mvc_weights_point_numba_out(
    p, cage_V, cage_F, lam, Ehat, En, eps=1e-8
):
    """
    Ju, Schaefer & Warren (2005) 3D MVC.

    Ehat is reused for unit directions and En for distances.
    Results are written into lam.
    """
    n = cage_V.shape[0]
    nf = cage_F.shape[0]

    # Zero output (callers reuse rows of a matrix).
    for i in range(n):
        lam[i] = 0.0

    min_dist = np.inf
    min_idx = 0

    # Distances and unit directions.
    for i in range(n):
        dx = cage_V[i, 0] - p[0]
        dy = cage_V[i, 1] - p[1]
        dz = cage_V[i, 2] - p[2]

        di = np.sqrt(dx * dx + dy * dy + dz * dz)
        En[i] = di

        if di < min_dist:
            min_dist = di
            min_idx = i

        inv = 1.0 / di if di > 0.0 else 0.0
        Ehat[i, 0] = dx * inv
        Ehat[i, 1] = dy * inv
        Ehat[i, 2] = dz * inv

    # Point coincides with a cage vertex.
    if min_dist < eps:
        lam[min_idx] = 1.0
        return

    total = 0.0

    for f in range(nf):
        i1 = cage_F[f, 0]
        i2 = cage_F[f, 1]
        i3 = cage_F[f, 2]

        u1x = Ehat[i1, 0]; u1y = Ehat[i1, 1]; u1z = Ehat[i1, 2]
        u2x = Ehat[i2, 0]; u2y = Ehat[i2, 1]; u2z = Ehat[i2, 2]
        u3x = Ehat[i3, 0]; u3y = Ehat[i3, 1]; u3z = Ehat[i3, 2]

        d1 = En[i1]
        d2 = En[i2]
        d3 = En[i3]

        # Spherical chord lengths.
        ex = u2x - u3x; ey = u2y - u3y; ez = u2z - u3z
        l1 = np.sqrt(ex * ex + ey * ey + ez * ez)
        dx = u3x - u1x; dy = u3y - u1y; dz = u3z - u1z
        l2 = np.sqrt(dx * dx + dy * dy + dz * dz)
        dx = u1x - u2x; dy = u1y - u2y; dz = u1z - u2z
        l3 = np.sqrt(dx * dx + dy * dy + dz * dz)

        a1 = 0.5 * l1
        if a1 > 1.0: a1 = 1.0
        elif a1 < -1.0: a1 = -1.0
        theta1 = 2.0 * np.arcsin(a1)

        a2 = 0.5 * l2
        if a2 > 1.0: a2 = 1.0
        elif a2 < -1.0: a2 = -1.0
        theta2 = 2.0 * np.arcsin(a2)

        a3 = 0.5 * l3
        if a3 > 1.0: a3 = 1.0
        elif a3 < -1.0: a3 = -1.0
        theta3 = 2.0 * np.arcsin(a3)

        h = 0.5 * (theta1 + theta2 + theta3)

        # Point lies on this triangular face.
        if np.pi - h < eps:
            w1 = np.sin(theta1) * d2 * d3
            w2 = np.sin(theta2) * d3 * d1
            w3 = np.sin(theta3) * d1 * d2
            tot = w1 + w2 + w3
            for k in range(n):
                lam[k] = 0.0
            lam[i1] = w1 / tot
            lam[i2] = w2 / tot
            lam[i3] = w3 / tot
            return

        sin_h = np.sin(h)
        sin_t1 = np.sin(theta1)
        sin_t2 = np.sin(theta2)
        sin_t3 = np.sin(theta3)

        c1 = (2.0 * sin_h * np.sin(h - theta1)) / (sin_t2 * sin_t3) - 1.0
        c2 = (2.0 * sin_h * np.sin(h - theta2)) / (sin_t3 * sin_t1) - 1.0
        c3 = (2.0 * sin_h * np.sin(h - theta3)) / (sin_t1 * sin_t2) - 1.0

        # det([u1; u2; u3]) inlined.
        det = (u1x * (u2y * u3z - u2z * u3y)
             - u1y * (u2x * u3z - u2z * u3x)
             + u1z * (u2x * u3y - u2y * u3x))
        sgn = 1.0 if det >= 0.0 else -1.0

        v1 = 1.0 - c1 * c1
        v2 = 1.0 - c2 * c2
        v3 = 1.0 - c3 * c3
        if v1 < 0.0: v1 = 0.0
        if v2 < 0.0: v2 = 0.0
        if v3 < 0.0: v3 = 0.0
        s1 = sgn * np.sqrt(v1)
        s2 = sgn * np.sqrt(v2)
        s3 = sgn * np.sqrt(v3)

        if abs(s1) < eps or abs(s2) < eps or abs(s3) < eps:
            # x on plane of f but outside the face -> ignore this face.
            continue

        w1 = (theta1 - c2 * theta3 - c3 * theta2) / (d1 * sin_t2 * s3)
        w2 = (theta2 - c3 * theta1 - c1 * theta3) / (d2 * sin_t3 * s1)
        w3 = (theta3 - c1 * theta2 - c2 * theta1) / (d3 * sin_t1 * s2)

        lam[i1] += w1
        lam[i2] += w2
        lam[i3] += w3
        total += w1 + w2 + w3

    inv = 1.0 / total
    for i in range(n):
        lam[i] *= inv


In [10]:
# @njit(cache=True, fastmath=False)
# def mvc_weights_point_numba_out(
#     p, cage_V, cage_F, w_out, Ehat, En, eps=1e-8
# ):
#     """
#     Ju, Schaefer & Warren (2005) 3D MVC.

#     Ehat is reused for unit directions and En for distances.
#     Results are written into w_out.
#     """
#     N = cage_V.shape[0]
#     K = cage_F.shape[0]

#     min_dist = np.inf
#     min_idx = 0

#     # Distances and unit directions.
#     for i in range(N):
#         ex = cage_V[i, 0] - p[0]
#         ey = cage_V[i, 1] - p[1]
#         ez = cage_V[i, 2] - p[2]

#         dist = np.sqrt(ex * ex + ey * ey + ez * ez)
#         En[i] = dist
#         w_out[i] = 0.0

#         if dist < min_dist:
#             min_dist = dist
#             min_idx = i

#         if dist > 0.0:
#             inv_dist = 1.0 / dist
#             Ehat[i, 0] = ex * inv_dist
#             Ehat[i, 1] = ey * inv_dist
#             Ehat[i, 2] = ez * inv_dist
#         else:
#             Ehat[i, 0] = 0.0
#             Ehat[i, 1] = 0.0
#             Ehat[i, 2] = 0.0

#     # Point coincides with a cage vertex.
#     if min_dist < eps:
#         w_out[min_idx] = 1.0
#         return

#     total = 0.0

#     for f in range(K):
#         i1 = cage_F[f, 0]
#         i2 = cage_F[f, 1]
#         i3 = cage_F[f, 2]

#         u1x = Ehat[i1, 0]
#         u1y = Ehat[i1, 1]
#         u1z = Ehat[i1, 2]

#         u2x = Ehat[i2, 0]
#         u2y = Ehat[i2, 1]
#         u2z = Ehat[i2, 2]

#         u3x = Ehat[i3, 0]
#         u3y = Ehat[i3, 1]
#         u3z = Ehat[i3, 2]

#         d1 = En[i1]
#         d2 = En[i2]
#         d3 = En[i3]

#         # Spherical chord lengths.
#         dx = u2x - u3x
#         dy = u2y - u3y
#         dz = u2z - u3z
#         l1 = np.sqrt(dx * dx + dy * dy + dz * dz)

#         dx = u3x - u1x
#         dy = u3y - u1y
#         dz = u3z - u1z
#         l2 = np.sqrt(dx * dx + dy * dy + dz * dz)

#         dx = u1x - u2x
#         dy = u1y - u2y
#         dz = u1z - u2z
#         l3 = np.sqrt(dx * dx + dy * dy + dz * dz)

#         a1 = 0.5 * l1
#         a2 = 0.5 * l2
#         a3 = 0.5 * l3

#         if a1 > 1.0:
#             a1 = 1.0
#         if a2 > 1.0:
#             a2 = 1.0
#         if a3 > 1.0:
#             a3 = 1.0

#         theta1 = 2.0 * np.arcsin(a1)
#         theta2 = 2.0 * np.arcsin(a2)
#         theta3 = 2.0 * np.arcsin(a3)

#         half_sum = 0.5 * (theta1 + theta2 + theta3)

#         # Point lies on this triangular face.
#         if np.pi - half_sum < eps:
#             w1 = np.sin(theta1) * d2 * d3
#             w2 = np.sin(theta2) * d3 * d1
#             w3 = np.sin(theta3) * d1 * d2
#             face_sum = w1 + w2 + w3

#             if abs(face_sum) > eps:
#                 for i in range(N):
#                     w_out[i] = 0.0

#                 w_out[i1] = w1 / face_sum
#                 w_out[i2] = w2 / face_sum
#                 w_out[i3] = w3 / face_sum
#                 return

#         sin_h = np.sin(half_sum)
#         sin_t1 = np.sin(theta1)
#         sin_t2 = np.sin(theta2)
#         sin_t3 = np.sin(theta3)

#         # Degenerate spherical triangle.
#         if (abs(sin_t1) < eps or
#                 abs(sin_t2) < eps or
#                 abs(sin_t3) < eps):
#             continue

#         c1 = (
#             2.0 * sin_h * np.sin(half_sum - theta1)
#             / (sin_t2 * sin_t3)
#             - 1.0
#         )
#         c2 = (
#             2.0 * sin_h * np.sin(half_sum - theta2)
#             / (sin_t3 * sin_t1)
#             - 1.0
#         )
#         c3 = (
#             2.0 * sin_h * np.sin(half_sum - theta3)
#             / (sin_t1 * sin_t2)
#             - 1.0
#         )

#         # Clamp roundoff before sqrt(1 - c^2).
#         if c1 > 1.0:
#             c1 = 1.0
#         elif c1 < -1.0:
#             c1 = -1.0

#         if c2 > 1.0:
#             c2 = 1.0
#         elif c2 < -1.0:
#             c2 = -1.0

#         if c3 > 1.0:
#             c3 = 1.0
#         elif c3 < -1.0:
#             c3 = -1.0

#         det = (
#             u1x * (u2y * u3z - u2z * u3y)
#             - u1y * (u2x * u3z - u2z * u3x)
#             + u1z * (u2x * u3y - u2y * u3x)
#         )

#         if det >= 0.0:
#             sign_det = 1.0
#         else:
#             sign_det = -1.0

#         s1 = sign_det * np.sqrt(max(0.0, 1.0 - c1 * c1))
#         s2 = sign_det * np.sqrt(max(0.0, 1.0 - c2 * c2))
#         s3 = sign_det * np.sqrt(max(0.0, 1.0 - c3 * c3))

#         # Coplanar but outside this face.
#         if abs(s1) < eps or abs(s2) < eps or abs(s3) < eps:
#             continue

#         w1 = (
#             theta1 - c2 * theta3 - c3 * theta2
#         ) / (d1 * sin_t2 * s3)

#         w2 = (
#             theta2 - c3 * theta1 - c1 * theta3
#         ) / (d2 * sin_t3 * s1)

#         w3 = (
#             theta3 - c1 * theta2 - c2 * theta1
#         ) / (d3 * sin_t1 * s2)

#         w_out[i1] += w1
#         w_out[i2] += w2
#         w_out[i3] += w3
#         total += w1 + w2 + w3

#     if abs(total) > eps:
#         inv_total = 1.0 / total
#         for i in range(N):
#             w_out[i] *= inv_total
#     else:
#         # Defensive fallback for degenerate/invalid cages.
#         for i in range(N):
#             w_out[i] = 0.0
#         w_out[min_idx] = 1.0

In [11]:
@njit(cache=True, fastmath=False)
def mvc_weights_point_numba_out(p, cage_V, cage_F, w_out, Ehat, En, eps=0.0):
    """
    Same MVC math as your mvc_weights_point_numba, but writes into w_out (preallocated).
    Ehat (N,3) and En (N,) are workspace buffers (also preallocated).
    """
    N = cage_V.shape[0]
    K = cage_F.shape[0]

    # Ehat, En
    for i in range(N):
        ex = cage_V[i, 0] - p[0]
        ey = cage_V[i, 1] - p[1]
        ez = cage_V[i, 2] - p[2]
        nrm = np.sqrt(ex*ex + ey*ey + ez*ez)
        En[i] = nrm
        inv = 1.0 / (nrm + eps)
        Ehat[i, 0] = ex * inv
        Ehat[i, 1] = ey * inv
        Ehat[i, 2] = ez * inv
        w_out[i] = 0.0  # zero weights here to avoid an extra loop

    # faces
    for f in range(K):
        j = cage_F[f, 0]
        k = cage_F[f, 1]
        l = cage_F[f, 2]

        Ejx, Ejy, Ejz = Ehat[j, 0], Ehat[j, 1], Ehat[j, 2]
        Ekx, Eky, Ekz = Ehat[k, 0], Ehat[k, 1], Ehat[k, 2]
        Elx, Ely, Elz = Ehat[l, 0], Ehat[l, 1], Ehat[l, 2]

        # det = dot(Ej, cross(Ek, El))
        cx = Eky*Elz - Ekz*Ely
        cy = Ekz*Elx - Ekx*Elz
        cz = Ekx*Ely - Eky*Elx
        det = Ejx*cx + Ejy*cy + Ejz*cz

        # of = sign(det)
        of = 0.0
        if det > 0.0:
            of = 1.0
        elif det < 0.0:
            of = -1.0

        # n_jk = unit_normal(Ej, Ek)
        cx = Ejy*Ekz - Ejz*Eky
        cy = Ejz*Ekx - Ejx*Ekz
        cz = Ejx*Eky - Ejy*Ekx
        cn = np.sqrt(cx*cx + cy*cy + cz*cz)
        inv = 1.0 / (cn + eps)
        n_jk_x = of * cx * inv
        n_jk_y = of * cy * inv
        n_jk_z = of * cz * inv

        # n_kl = unit_normal(Ek, El)
        cx = Eky*Elz - Ekz*Ely
        cy = Ekz*Elx - Ekx*Elz
        cz = Ekx*Ely - Eky*Elx
        cn = np.sqrt(cx*cx + cy*cy + cz*cz)
        inv = 1.0 / (cn + eps)
        n_kl_x = of * cx * inv
        n_kl_y = of * cy * inv
        n_kl_z = of * cz * inv

        # n_lj = unit_normal(El, Ej)
        cx = Ely*Ejz - Elz*Ejy
        cy = Elz*Ejx - Elx*Ejz
        cz = Elx*Ejy - Ely*Ejx
        cn = np.sqrt(cx*cx + cy*cy + cz*cz)
        inv = 1.0 / (cn + eps)
        n_lj_x = of * cx * inv
        n_lj_y = of * cy * inv
        n_lj_z = of * cz * inv

        # angles (no clipping)
        d = Ejx*Ekx + Ejy*Eky + Ejz*Ekz
        th_jk = np.arccos(d)
        d = Ekx*Elx + Eky*Ely + Ekz*Elz
        th_kl = np.arccos(d)
        d = Elx*Ejx + Ely*Ejy + Elz*Ejz
        th_lj = np.arccos(d)

        # m_f
        mf_x = 0.5 * (th_jk*n_jk_x + th_kl*n_kl_x + th_lj*n_lj_x)
        mf_y = 0.5 * (th_jk*n_jk_y + th_kl*n_kl_y + th_lj*n_lj_y)
        mf_z = 0.5 * (th_jk*n_jk_z + th_kl*n_kl_z + th_lj*n_lj_z)

        # mu_j (n_kl, Ej)
        num = n_kl_x*mf_x + n_kl_y*mf_y + n_kl_z*mf_z
        den = n_kl_x*Ejx  + n_kl_y*Ejy  + n_kl_z*Ejz
        mu_j = num / (den + eps)

        # mu_k (n_lj, Ek)
        num = n_lj_x*mf_x + n_lj_y*mf_y + n_lj_z*mf_z
        den = n_lj_x*Ekx  + n_lj_y*Eky  + n_lj_z*Ekz
        mu_k = num / (den + eps)

        # mu_l (n_jk, El)
        num = n_jk_x*mf_x + n_jk_y*mf_y + n_jk_z*mf_z
        den = n_jk_x*Elx  + n_jk_y*Ely  + n_jk_z*Elz
        mu_l = num / (den + eps)

        cj = of * mu_j / (En[j] + eps)
        ck = of * mu_k / (En[k] + eps)
        cl = of * mu_l / (En[l] + eps)

        w_out[j] += cj
        w_out[k] += ck
        w_out[l] += cl

    # normalize
    s = 0.0
    for i in range(N):
        s += w_out[i]
    invs = 1.0 / (s + eps)
    for i in range(N):
        w_out[i] *= invs

In [12]:
@njit(cache=True, fastmath=False)
def solve_3x3(A, b):
    M = np.empty((3, 4), dtype=np.float64)
    for i in range(3):
        M[i, 0] = A[i, 0]; M[i, 1] = A[i, 1]; M[i, 2] = A[i, 2]; M[i, 3] = b[i]

    for k in range(3):
        piv = k
        maxabs = abs(M[k, k])
        for r in range(k + 1, 3):
            v = abs(M[r, k])
            if v > maxabs:
                maxabs = v
                piv = r
        if piv != k:
            for j in range(k, 4):
                tmp = M[k, j]; M[k, j] = M[piv, j]; M[piv, j] = tmp

        pivv = M[k, k]
        invp = 1.0 / pivv
        for j in range(k, 4):
            M[k, j] *= invp

        for r in range(3):
            if r == k:
                continue
            factor = M[r, k]
            for j in range(k, 4):
                M[r, j] -= factor * M[k, j]

    x = np.empty(3, dtype=np.float64)
    x[0] = M[0, 3]; x[1] = M[1, 3]; x[2] = M[2, 3]
    return x


In [14]:
V_seq_c.shape, V_seq_c[:, 0, 0].shape, U_seq_c.shape

((150, 10002, 3), (150,), (150, 2502, 3))

In [ ]:
recreated_mesh = trimesh.Trimesh(vertices=refined_mesh_vertices, faces=mesh_faces)


In [ ]:
@njit(cache=True, fastmath=False)
def mvc_weights_point_numba_out(
    p, cage_V, cage_F, lam, Ehat=None, En=None, eps=1e-8
):
    """
    Ju, Schaefer & Warren (2005) 3D MVC.

    Ehat is reused for unit directions and En for distances.
    Results are written into lam.
    """
    n = cage_V.shape[0]
    nf = cage_F.shape[0]

    # Zero output (callers reuse rows of a matrix).
    for i in range(n):
        lam[i] = 0.0

    min_dist = np.inf
    min_idx = 0

    # Distances and unit directions.
    for i in range(n):
        dx = cage_V[i, 0] - p[0]
        dy = cage_V[i, 1] - p[1]
        dz = cage_V[i, 2] - p[2]

        di = np.sqrt(dx * dx + dy * dy + dz * dz)
        En[i] = di

        if di < min_dist:
            min_dist = di
            min_idx = i

        inv = 1.0 / di if di > 0.0 else 0.0
        Ehat[i, 0] = dx * inv
        Ehat[i, 1] = dy * inv
        Ehat[i, 2] = dz * inv

    # Point coincides with a cage vertex.
    if min_dist < eps:
        lam[min_idx] = 1.0
        return

    total = 0.0

    for f in range(nf):
        i1 = cage_F[f, 0]
        i2 = cage_F[f, 1]
        i3 = cage_F[f, 2]

        u1x = Ehat[i1, 0]; u1y = Ehat[i1, 1]; u1z = Ehat[i1, 2]
        u2x = Ehat[i2, 0]; u2y = Ehat[i2, 1]; u2z = Ehat[i2, 2]
        u3x = Ehat[i3, 0]; u3y = Ehat[i3, 1]; u3z = Ehat[i3, 2]

        d1 = En[i1]
        d2 = En[i2]
        d3 = En[i3]

        # Spherical chord lengths.
        ex = u2x - u3x; ey = u2y - u3y; ez = u2z - u3z
        l1 = np.sqrt(ex * ex + ey * ey + ez * ez)
        dx = u3x - u1x; dy = u3y - u1y; dz = u3z - u1z
        l2 = np.sqrt(dx * dx + dy * dy + dz * dz)
        dx = u1x - u2x; dy = u1y - u2y; dz = u1z - u2z
        l3 = np.sqrt(dx * dx + dy * dy + dz * dz)

        a1 = 0.5 * l1
        if a1 > 1.0: a1 = 1.0
        elif a1 < -1.0: a1 = -1.0
        theta1 = 2.0 * np.arcsin(a1)

        a2 = 0.5 * l2
        if a2 > 1.0: a2 = 1.0
        elif a2 < -1.0: a2 = -1.0
        theta2 = 2.0 * np.arcsin(a2)

        a3 = 0.5 * l3
        if a3 > 1.0: a3 = 1.0
        elif a3 < -1.0: a3 = -1.0
        theta3 = 2.0 * np.arcsin(a3)

        h = 0.5 * (theta1 + theta2 + theta3)

        # Point lies on this triangular face.
        if np.pi - h < eps:
            w1 = np.sin(theta1) * d2 * d3
            w2 = np.sin(theta2) * d3 * d1
            w3 = np.sin(theta3) * d1 * d2
            tot = w1 + w2 + w3
            for k in range(n):
                lam[k] = 0.0
            lam[i1] = w1 / tot
            lam[i2] = w2 / tot
            lam[i3] = w3 / tot
            return

        sin_h = np.sin(h)
        sin_t1 = np.sin(theta1)
        sin_t2 = np.sin(theta2)
        sin_t3 = np.sin(theta3)

        c1 = (2.0 * sin_h * np.sin(h - theta1)) / (sin_t2 * sin_t3) - 1.0
        c2 = (2.0 * sin_h * np.sin(h - theta2)) / (sin_t3 * sin_t1) - 1.0
        c3 = (2.0 * sin_h * np.sin(h - theta3)) / (sin_t1 * sin_t2) - 1.0

        # det([u1; u2; u3]) inlined.
        det = (u1x * (u2y * u3z - u2z * u3y)
             - u1y * (u2x * u3z - u2z * u3x)
             + u1z * (u2x * u3y - u2y * u3x))
        sgn = 1.0 if det >= 0.0 else -1.0

        v1 = 1.0 - c1 * c1
        v2 = 1.0 - c2 * c2
        v3 = 1.0 - c3 * c3
        if v1 < 0.0: v1 = 0.0
        if v2 < 0.0: v2 = 0.0
        if v3 < 0.0: v3 = 0.0
        s1 = sgn * np.sqrt(v1)
        s2 = sgn * np.sqrt(v2)
        s3 = sgn * np.sqrt(v3)

        if abs(s1) < eps or abs(s2) < eps or abs(s3) < eps:
            # x on plane of f but outside the face -> ignore this face.
            continue

        w1 = (theta1 - c2 * theta3 - c3 * theta2) / (d1 * sin_t2 * s3)
        w2 = (theta2 - c3 * theta1 - c1 * theta3) / (d2 * sin_t3 * s1)
        w3 = (theta3 - c1 * theta2 - c2 * theta1) / (d3 * sin_t1 * s2)

        lam[i1] += w1
        lam[i2] += w2
        lam[i3] += w3
        total += w1 + w2 + w3

    inv = 1.0 / total
    for i in range(n):
        lam[i] *= inv


In [30]:
@njit(cache=True, fastmath=False)
def mvc_weights_point_numba_out(p, cage_V, cage_F, w_out, Ehat, En, eps=0.0):
    """
    Same MVC math as your mvc_weights_point_numba, but writes into w_out (preallocated).
    Ehat (N,3) and En (N,) are workspace buffers (also preallocated).
    """
    N = cage_V.shape[0]
    K = cage_F.shape[0]

    # Ehat, En
    for i in range(N):
        ex = cage_V[i, 0] - p[0]
        ey = cage_V[i, 1] - p[1]
        ez = cage_V[i, 2] - p[2]
        nrm = np.sqrt(ex*ex + ey*ey + ez*ez)
        En[i] = nrm
        inv = 1.0 / (nrm + eps)
        Ehat[i, 0] = ex * inv
        Ehat[i, 1] = ey * inv
        Ehat[i, 2] = ez * inv
        w_out[i] = 0.0  # zero weights here to avoid an extra loop

    # faces
    for f in range(K):
        j = cage_F[f, 0]
        k = cage_F[f, 1]
        l = cage_F[f, 2]

        Ejx, Ejy, Ejz = Ehat[j, 0], Ehat[j, 1], Ehat[j, 2]
        Ekx, Eky, Ekz = Ehat[k, 0], Ehat[k, 1], Ehat[k, 2]
        Elx, Ely, Elz = Ehat[l, 0], Ehat[l, 1], Ehat[l, 2]

        # det = dot(Ej, cross(Ek, El))
        cx = Eky*Elz - Ekz*Ely
        cy = Ekz*Elx - Ekx*Elz
        cz = Ekx*Ely - Eky*Elx
        det = Ejx*cx + Ejy*cy + Ejz*cz

        # of = sign(det)
        of = 0.0
        if det > 0.0:
            of = 1.0
        elif det < 0.0:
            of = -1.0

        # n_jk = unit_normal(Ej, Ek)
        cx = Ejy*Ekz - Ejz*Eky
        cy = Ejz*Ekx - Ejx*Ekz
        cz = Ejx*Eky - Ejy*Ekx
        cn = np.sqrt(cx*cx + cy*cy + cz*cz)
        inv = 1.0 / (cn + eps)
        n_jk_x = of * cx * inv
        n_jk_y = of * cy * inv
        n_jk_z = of * cz * inv

        # n_kl = unit_normal(Ek, El)
        cx = Eky*Elz - Ekz*Ely
        cy = Ekz*Elx - Ekx*Elz
        cz = Ekx*Ely - Eky*Elx
        cn = np.sqrt(cx*cx + cy*cy + cz*cz)
        inv = 1.0 / (cn + eps)
        n_kl_x = of * cx * inv
        n_kl_y = of * cy * inv
        n_kl_z = of * cz * inv

        # n_lj = unit_normal(El, Ej)
        cx = Ely*Ejz - Elz*Ejy
        cy = Elz*Ejx - Elx*Ejz
        cz = Elx*Ejy - Ely*Ejx
        cn = np.sqrt(cx*cx + cy*cy + cz*cz)
        inv = 1.0 / (cn + eps)
        n_lj_x = of * cx * inv
        n_lj_y = of * cy * inv
        n_lj_z = of * cz * inv

        # angles (no clipping)
        d = Ejx*Ekx + Ejy*Eky + Ejz*Ekz
        th_jk = np.arccos(d)
        d = Ekx*Elx + Eky*Ely + Ekz*Elz
        th_kl = np.arccos(d)
        d = Elx*Ejx + Ely*Ejy + Elz*Ejz
        th_lj = np.arccos(d)

        # m_f
        mf_x = 0.5 * (th_jk*n_jk_x + th_kl*n_kl_x + th_lj*n_lj_x)
        mf_y = 0.5 * (th_jk*n_jk_y + th_kl*n_kl_y + th_lj*n_lj_y)
        mf_z = 0.5 * (th_jk*n_jk_z + th_kl*n_kl_z + th_lj*n_lj_z)

        # mu_j (n_kl, Ej)
        num = n_kl_x*mf_x + n_kl_y*mf_y + n_kl_z*mf_z
        den = n_kl_x*Ejx  + n_kl_y*Ejy  + n_kl_z*Ejz
        mu_j = num / (den + eps)

        # mu_k (n_lj, Ek)
        num = n_lj_x*mf_x + n_lj_y*mf_y + n_lj_z*mf_z
        den = n_lj_x*Ekx  + n_lj_y*Eky  + n_lj_z*Ekz
        mu_k = num / (den + eps)

        # mu_l (n_jk, El)
        num = n_jk_x*mf_x + n_jk_y*mf_y + n_jk_z*mf_z
        den = n_jk_x*Elx  + n_jk_y*Ely  + n_jk_z*Elz
        mu_l = num / (den + eps)

        cj = of * mu_j / (En[j] + eps)
        ck = of * mu_k / (En[k] + eps)
        cl = of * mu_l / (En[l] + eps)

        w_out[j] += cj
        w_out[k] += ck
        w_out[l] += cl

    # normalize
    s = 0.0
    for i in range(N):
        s += w_out[i]
    invs = 1.0 / (s + eps)
    for i in range(N):
        w_out[i] *= invs

***From Here***

In [3]:
@njit(cache=True, fastmath=True)
def mvc_3d_single_into(x, V, F, eps, lam, Ehat, En):
    """
    Compute 3D MVC weights of point x w.r.t. mesh (V, F) into lam (length n).
    lam is zeroed inside; on a vertex coincidence or in-face hit, only the
    relevant entries are nonzero.
    """
    n = V.shape[0]
    nf = F.shape[0]

    # Zero output (callers reuse rows of a matrix).
    for i in range(n):
        lam[i] = 0.0

    # Per-vertex distances and unit directions from x.
    if Ehat is None:
        Ehat = np.empty((n, 3))
    if En is None:
        En = np.empty(n)
    # dist = np.empty(n)
    # u = np.empty((n, 3))
    min_dist = np.inf
    min_idx = 0
    for i in range(n):
        dx = V[i, 0] - x[0]
        dy = V[i, 1] - x[1]
        dz = V[i, 2] - x[2]
        di = np.sqrt(dx * dx + dy * dy + dz * dz)
        En[i] = di
        if di < min_dist:
            min_dist = di
            min_idx = i
        inv = 1.0 / di if di > 0.0 else 0.0
        Ehat[i, 0] = dx * inv
        Ehat[i, 1] = dy * inv
        Ehat[i, 2] = dz * inv

    # Coincides with a cage vertex -> Kronecker delta.
    if min_dist < eps:
        lam[min_idx] = 1.0
        return

    total = 0.0
    for fi in range(nf):
        i1 = F[fi, 0]
        i2 = F[fi, 1]
        i3 = F[fi, 2]

        u1x = Ehat[i1, 0]; u1y = Ehat[i1, 1]; u1z = Ehat[i1, 2]
        u2x = Ehat[i2, 0]; u2y = Ehat[i2, 1]; u2z = Ehat[i2, 2]
        u3x = Ehat[i3, 0]; u3y = Ehat[i3, 1]; u3z = Ehat[i3, 2]

        d1 = En[i1]
        d2 = En[i2]
        d3 = En[i3]

        # Chord lengths between u_i, u_j on S^2.
        ex = u2x - u3x; ey = u2y - u3y; ez = u2z - u3z
        l1 = np.sqrt(ex * ex + ey * ey + ez * ez)
        ex = u3x - u1x; ey = u3y - u1y; ez = u3z - u1z
        l2 = np.sqrt(ex * ex + ey * ey + ez * ez)
        ex = u1x - u2x; ey = u1y - u2y; ez = u1z - u2z
        l3 = np.sqrt(ex * ex + ey * ey + ez * ez)

        a1 = 0.5 * l1
        if a1 > 1.0: a1 = 1.0
        elif a1 < -1.0: a1 = -1.0
        theta1 = 2.0 * np.arcsin(a1)

        a2 = 0.5 * l2
        if a2 > 1.0: a2 = 1.0
        elif a2 < -1.0: a2 = -1.0
        theta2 = 2.0 * np.arcsin(a2)

        a3 = 0.5 * l3
        if a3 > 1.0: a3 = 1.0
        elif a3 < -1.0: a3 = -1.0
        theta3 = 2.0 * np.arcsin(a3)

        h = 0.5 * (theta1 + theta2 + theta3)

        # x lies on the plane of f, inside f -> 2D barycentric on this face.
        if np.pi - h < eps:
            w1 = np.sin(theta1) * d2 * d3
            w2 = np.sin(theta2) * d3 * d1
            w3 = np.sin(theta3) * d1 * d2
            tot = w1 + w2 + w3
            for k in range(n):
                lam[k] = 0.0
            lam[i1] = w1 / tot
            lam[i2] = w2 / tot
            lam[i3] = w3 / tot
            return

        sin_h = np.sin(h)
        sin_t1 = np.sin(theta1)
        sin_t2 = np.sin(theta2)
        sin_t3 = np.sin(theta3)

        c1 = (2.0 * sin_h * np.sin(h - theta1)) / (sin_t2 * sin_t3) - 1.0
        c2 = (2.0 * sin_h * np.sin(h - theta2)) / (sin_t3 * sin_t1) - 1.0
        c3 = (2.0 * sin_h * np.sin(h - theta3)) / (sin_t1 * sin_t2) - 1.0

        # det([u1; u2; u3]) inlined.
        det = (u1x * (u2y * u3z - u2z * u3y)
             - u1y * (u2x * u3z - u2z * u3x)
             + u1z * (u2x * u3y - u2y * u3x))
        sgn = 1.0 if det >= 0.0 else -1.0

        v1 = 1.0 - c1 * c1
        v2 = 1.0 - c2 * c2
        v3 = 1.0 - c3 * c3
        if v1 < 0.0: v1 = 0.0
        if v2 < 0.0: v2 = 0.0
        if v3 < 0.0: v3 = 0.0
        s1 = sgn * np.sqrt(v1)
        s2 = sgn * np.sqrt(v2)
        s3 = sgn * np.sqrt(v3)

        if abs(s1) < eps or abs(s2) < eps or abs(s3) < eps:
            # x on plane of f but outside the face -> ignore this face.
            continue

        w1 = (theta1 - c2 * theta3 - c3 * theta2) / (d1 * sin_t2 * s3)
        w2 = (theta2 - c3 * theta1 - c1 * theta3) / (d2 * sin_t3 * s1)
        w3 = (theta3 - c1 * theta2 - c2 * theta1) / (d3 * sin_t1 * s2)

        lam[i1] += w1
        lam[i2] += w2
        lam[i3] += w3
        total += w1 + w2 + w3

    inv = 1.0 / total
    for i in range(n):
        lam[i] *= inv


@njit(cache=True, fastmath=True)
def mvc_weights_point_numba_out(p, cage_V, cage_F, lam, Ehat=None, En=None, eps=1e-8):
    """Allocating wrapper, matches the original signature."""
    lam = np.empty(cage_V.shape[0])
    mvc_3d_single_into(p, cage_V, cage_F, eps, lam, Ehat=Ehat, En=En)
    return lam

In [4]:
import numpy as np
from numba import njit
        
@njit(cache=True, fastmath=False)
def solve_3x3(A, b):
    M = np.empty((3, 4), dtype=np.float64)
    for i in range(3):
        M[i, 0] = A[i, 0]; M[i, 1] = A[i, 1]; M[i, 2] = A[i, 2]; M[i, 3] = b[i]

    for k in range(3):
        piv = k
        maxabs = abs(M[k, k])
        for r in range(k + 1, 3):
            v = abs(M[r, k])
            if v > maxabs:
                maxabs = v
                piv = r
        if piv != k:
            for j in range(k, 4):
                tmp = M[k, j]; M[k, j] = M[piv, j]; M[piv, j] = tmp

        pivv = M[k, k]
        invp = 1.0 / pivv
        for j in range(k, 4):
            M[k, j] *= invp

        for r in range(3):
            if r == k:
                continue
            factor = M[r, k]
            for j in range(k, 4):
                M[r, j] -= factor * M[k, j]

    x = np.empty(3, dtype=np.float64)
    x[0] = M[0, 3]; x[1] = M[1, 3]; x[2] = M[2, 3]
    return x


@njit(cache=True, fastmath=False)
def refine_vertex_point_newton_method_numba_fast(
    p0_np, U_seq_np, V_seq_i_np, cage_V0_np, cage_F_np,
    steps=1, alpha=0.0, h=1e-9, mvc_eps=0.0
):
    p = p0_np.copy()

    m = U_seq_np.shape[0]
    N = U_seq_np.shape[1]

    # b = sum_l U_l v_l  -> (N,)
    b = np.zeros(N, dtype=np.float64)
    for l in range(m):
        vx = V_seq_i_np[l, 0]
        vy = V_seq_i_np[l, 1]
        vz = V_seq_i_np[l, 2]
        for n in range(N):
            b[n] += (U_seq_np[l, n, 0] * vx +
                     U_seq_np[l, n, 1] * vy +
                     U_seq_np[l, n, 2] * vz)

    # work buffers (allocated once)
    w     = np.empty(N, dtype=np.float64)
    w_pos = np.empty(N, dtype=np.float64)

    Ehat = np.empty((N, 3), dtype=np.float64)
    En   = np.empty(N, dtype=np.float64)

    Ehat2 = np.empty((N, 3), dtype=np.float64)  # for w_pos calls
    En2   = np.empty(N, dtype=np.float64)

    D   = np.empty((N, 3), dtype=np.float64)
    pred = np.empty((m, 3), dtype=np.float64)
    Aw  = np.empty(N, dtype=np.float64)
    r   = np.empty(N, dtype=np.float64)

    T   = np.empty((m, 3, 3), dtype=np.float64)
    AD  = np.empty((N, 3), dtype=np.float64)

    F   = np.empty(3, dtype=np.float64)
    DF  = np.empty((3, 3), dtype=np.float64)

    pp = np.empty(3, dtype=np.float64)

    for _ in range(steps):
        # w = MVC(p) into preallocated buffer
        mvc_weights_point_numba_out(p, cage_V0_np, cage_F_np, w, Ehat, En, mvc_eps)

        # D via forward finite differences
        invh = 1.0 / h
        for d in range(3):
            pp[0] = p[0]; pp[1] = p[1]; pp[2] = p[2]
            pp[d] += h
            mvc_weights_point_numba_out(pp, cage_V0_np, cage_F_np, w_pos, Ehat2, En2, mvc_eps)
            for n in range(N):
                D[n, d] = (w_pos[n] - w[n]) * invh

        # pred[l] = U_l^T w
        for l in range(m):
            s0 = 0.0; s1 = 0.0; s2 = 0.0
            for n in range(N):
                wn = w[n]
                s0 += U_seq_np[l, n, 0] * wn
                s1 += U_seq_np[l, n, 1] * wn
                s2 += U_seq_np[l, n, 2] * wn
            pred[l, 0] = s0; pred[l, 1] = s1; pred[l, 2] = s2

        # Aw = sum_l U_l pred[l]
        for n in range(N):
            s = 0.0
            for l in range(m):
                s += (U_seq_np[l, n, 0] * pred[l, 0] +
                      U_seq_np[l, n, 1] * pred[l, 1] +
                      U_seq_np[l, n, 2] * pred[l, 2])
            Aw[n] = s

        # r = Aw - b
        for n in range(N):
            r[n] = Aw[n] - b[n]

        # F = D^T r
        for d in range(3):
            s = 0.0
            for n in range(N):
                s += D[n, d] * r[n]
            F[d] = s

        # T[l] = U_l^T D   (3x3)
        for l in range(m):
            for c in range(3):
                for d in range(3):
                    s = 0.0
                    for n in range(N):
                        s += U_seq_np[l, n, c] * D[n, d]
                    T[l, c, d] = s

        # AD = sum_l U_l T[l]
        for n in range(N):
            for d in range(3):
                s = 0.0
                for l in range(m):
                    s += (U_seq_np[l, n, 0] * T[l, 0, d] +
                          U_seq_np[l, n, 1] * T[l, 1, d] +
                          U_seq_np[l, n, 2] * T[l, 2, d])
                AD[n, d] = s

        # DF = D^T AD (+ alpha I)
        for i in range(3):
            for j in range(3):
                s = 0.0
                for n in range(N):
                    s += D[n, i] * AD[n, j]
                DF[i, j] = s
        DF[0, 0] += alpha
        DF[1, 1] += alpha
        DF[2, 2] += alpha

        # dp = -solve(DF, F)
        x = solve_3x3(DF, F)
        p[0] -= x[0]
        p[1] -= x[1]
        p[2] -= x[2]

    return p


x = time.time()

# parse through all the .obj files in cage_avg/bouncing_cage_avg and create (m, N, 3) array of cage vertices
# cage_avg_folder = f"cage_avg/{action}_cage_avg"
cage_avg_folder = f"Dynamic mesh codec for mac v2/decompressed_{action}_1/{alpha_str}"
cage_files = sorted([f for f in os.listdir(cage_avg_folder) if f.endswith(".obj")])
cage_vertices_list = []
for cage_file in cage_files:
    cage_path = os.path.join(cage_avg_folder, cage_file)
    cage_V, cage_F = igl.read_triangle_mesh(cage_path)
    cage_vertices_list.append(cage_V)

U_seq_np = np.array(cage_vertices_list, dtype=np.float64)  # (m, N, 3)



# parse through all the .obj files in original_meshes/bouncing_mesh_off and create (m, N, 3) array of original mesh vertices
original_mesh_folder = f"original_meshes/{action}_mesh"
original_mesh_files = sorted([f for f in os.listdir(original_mesh_folder) if f.endswith(".obj")])
original_mesh_vertices_list = []
for original_mesh_file in original_mesh_files:
    original_mesh_path = os.path.join(original_mesh_folder, original_mesh_file)
    original_mesh_V, original_mesh_F = igl.read_triangle_mesh(original_mesh_path)
    original_mesh_vertices_list.append(original_mesh_V)

V_seq_np = np.array(original_mesh_vertices_list, dtype=np.float64)  # (m, N, 3)
# V_seq_np.shape


# input_file.replace('.obj', '_decoded_restructured_old.obj')


mesh_vertices, mesh_faces = igl.read_triangle_mesh(input_file.replace('.obj', '_decoded_restructured.obj'))


cage_vertices, cage_faces = igl.read_triangle_mesh(input_file.replace('.obj', '_cage_decoded_restructured.obj'))


# loop over all vertices and refine them
num_vertices = mesh_vertices.shape[0]
refined_mesh_vertices = np.zeros_like(mesh_vertices, dtype=np.float64)
# loss_per_vertex = np.zeros(num_vertices, dtype=np.float64)

mesh_vertices_c = np.ascontiguousarray(mesh_vertices, dtype=np.float64)
U_seq_c         = np.ascontiguousarray(U_seq_np, dtype=np.float64)
V_seq_c         = np.ascontiguousarray(V_seq_np, dtype=np.float64)
cage_V_c        = np.ascontiguousarray(cage_vertices, dtype=np.float64)
cage_F_c        = np.ascontiguousarray(cage_faces, dtype=np.int32)   # faces int32
refined_mesh_vertices = np.empty_like(mesh_vertices_c)



m = V_seq_c.shape[0]
V_i = np.empty((m, 3), dtype=np.float64)  # reused buffer

for i in range(mesh_vertices_c.shape[0]):
    print(f"Refining vertex {i+1}/{mesh_vertices_c.shape[0]}")

    # p0_np should be contiguous (copy is tiny, 3 floats)
    p0 = mesh_vertices_c[i].copy()

    # Make V_seq_i contiguous by copying into V_i (reused)
    V_i[:, 0] = V_seq_c[:, i, 0]
    V_i[:, 1] = V_seq_c[:, i, 1]
    V_i[:, 2] = V_seq_c[:, i, 2]

    refined_p = refine_vertex_point_newton_method_numba_fast(
        p0, U_seq_c, V_i,
        cage_V_c, cage_F_c,
        steps=1, alpha=0.0, h=1e-9
    )
    refined_mesh_vertices[i, :] = refined_p


y = time.time()
# print(f"Refinement took {t2 - t1:.2f} seconds")

e = y - x

recreated_mesh = trimesh.Trimesh(vertices=refined_mesh_vertices, faces=mesh_faces)


  o mesh_0001.off
  o mesh_0002.off
  o mesh_0003.off
  o mesh_0004.off
  o mesh_0005.off
  o mesh_0006.off
  o mesh_0007.off
  o mesh_0008.off
  o mesh_0009.off
  o mesh_0010.off
  o mesh_0011.off
  o mesh_0012.off
  o mesh_0013.off
  o mesh_0014.off
  o mesh_0015.off
  o mesh_0016.off
  o mesh_0017.off
  o mesh_0018.off
  o mesh_0019.off
  o mesh_0020.off
  o mesh_0021.off
  o mesh_0022.off
  o mesh_0023.off
  o mesh_0024.off
  o mesh_0025.off
  o mesh_0026.off
  o mesh_0027.off
  o mesh_0028.off
  o mesh_0029.off
  o mesh_0030.off
  o mesh_0031.off
  o mesh_0032.off
  o mesh_0033.off
  o mesh_0034.off
  o mesh_0035.off
  o mesh_0036.off
  o mesh_0037.off
  o mesh_0038.off
  o mesh_0039.off
  o mesh_0040.off
  o mesh_0041.off
  o mesh_0042.off
  o mesh_0043.off
  o mesh_0044.off
  o mesh_0045.off
  o mesh_0046.off
  o mesh_0047.off
  o mesh_0048.off
  o mesh_0049.off
  o mesh_0050.off
  o mesh_0051.off
  o mesh_0052.off
  o mesh_0053.off
  o mesh_0054.off
  o mesh_0055.off
  o mesh_0

Refining vertex 1/10002
Refining vertex 2/10002
Refining vertex 3/10002
Refining vertex 4/10002
Refining vertex 5/10002
Refining vertex 6/10002
Refining vertex 7/10002
Refining vertex 8/10002
Refining vertex 9/10002
Refining vertex 10/10002
Refining vertex 11/10002
Refining vertex 12/10002
Refining vertex 13/10002
Refining vertex 14/10002
Refining vertex 15/10002
Refining vertex 16/10002
Refining vertex 17/10002
Refining vertex 18/10002
Refining vertex 19/10002
Refining vertex 20/10002
Refining vertex 21/10002
Refining vertex 22/10002
Refining vertex 23/10002
Refining vertex 24/10002
Refining vertex 25/10002
Refining vertex 26/10002
Refining vertex 27/10002
Refining vertex 28/10002
Refining vertex 29/10002
Refining vertex 30/10002
Refining vertex 31/10002
Refining vertex 32/10002
Refining vertex 33/10002
Refining vertex 34/10002
Refining vertex 35/10002
Refining vertex 36/10002
Refining vertex 37/10002
Refining vertex 38/10002
Refining vertex 39/10002
Refining vertex 40/10002
Refining 

In [5]:
from binding.floater_mvc_fixed import compute_mvc as mvc_floater_fixed
from binding.floater_mvc_original import compute_mvc as mvc_floater
from binding.ju_mvc import compute_mvc_matrix as mvc_ju

/opt/homebrew/anaconda3/envs/nerf/lib/python3.12/site-packages/numba/core/decorators.py:246: RuntimeWarning: nopython is set for njit and is ignored
  warnings.warn('nopython is set for njit and is ignored', RuntimeWarning)


In [6]:
mvc = mvc_ju(mesh_vertices, cage_vertices, cage_faces)

In [7]:
mvc_1 = mvc_ju(recreated_mesh.vertices, cage_vertices, cage_faces)

In [8]:
cage_vertices_1, cage_faces_1 = igl.read_triangle_mesh("Dynamic mesh codec for mac v2/decompressed_jumping_1/0_25/cage_00008.obj")
mesh_vertices_1, mesh_faces_1 = igl.read_triangle_mesh("original_meshes/jumping_mesh/mesh_0008.obj")

  o mesh_0008.off


In [9]:
np.linalg.norm(mvc @ cage_vertices_1 - mesh_vertices_1)

np.float64(3.1556462015795557)

In [47]:
trimesh.Trimesh(vertices=mvc_1 @ cage_vertices_1, faces=mesh_faces_1).show()

In [10]:
np.linalg.norm(mvc_1 @ cage_vertices_1 - mesh_vertices_1)

np.float64(3.1456751409841326)